# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [64]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from tqdm import tqdm
from sklearn.metrics import accuracy_score

In [65]:
#sudo apt install python3-tqdm ## в терминале

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore).
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [66]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')

In [67]:
df

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [68]:
df_with_target = pd.read_csv('../data/dayofweek.csv')

In [69]:
df['dayofweek'] = df_with_target['dayofweek']

In [70]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [72]:
svc = SVC(random_state=21, probability=True)
params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    # 'C': (0.01, 0.1, 1, 1.5, 5, 10),
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}
grid = GridSearchCV(estimator=svc, 
                    param_grid=params,
                    cv=3, 
                    scoring='accuracy', 
                    verbose=2)

In [73]:
grid.fit(X_train, y_train)
print(f'Best score: {grid.best_score_}')
print(f'Best parameters: {grid.best_params_}')

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END ..class_weight=balanced, gamma=scale, kernel=linear; total time=  17.2s
[CV] END ..class_weight=balanced, gamma=scale, kernel=linear; total time=  20.6s
[CV] END ..class_weight=balanced, gamma=scale, kernel=linear; total time=  17.5s
[CV] END .....class_weight=balanced, gamma=scale, kernel=rbf; total time=   1.0s
[CV] END .....class_weight=balanced, gamma=scale, kernel=rbf; total time=   0.9s
[CV] END .....class_weight=balanced, gamma=scale, kernel=rbf; total time=   0.9s
[CV] END .class_weight=balanced, gamma=scale, kernel=sigmoid; total time=   0.9s
[CV] END .class_weight=balanced, gamma=scale, kernel=sigmoid; total time=   0.9s
[CV] END .class_weight=balanced, gamma=scale, kernel=sigmoid; total time=   1.0s
[CV] END ...class_weight=balanced, gamma=auto, kernel=linear; total time=  15.2s
[CV] END ...class_weight=balanced, gamma=auto, kernel=linear; total time=  21.2s
[CV] END ...class_weight=balanced, gamma=auto, k

Best score: 0.6595034232450713

Best parameters: {'class_weight': None, 'gamma': 'scale', 'kernel': 'linear'}

In [74]:
grid.cv_results_

{'mean_fit_time': array([18.37819425,  0.87733857,  0.89557385, 18.28036729,  0.86834566,
         1.11382445, 19.52340603,  0.68530877,  0.69576128, 15.67360083,
         0.74310851,  0.79575229]),
 'std_fit_time': array([1.54576789, 0.013361  , 0.03412981, 2.45702962, 0.01556165,
        0.01128928, 3.77156968, 0.00409617, 0.01314543, 4.00250942,
        0.0047715 , 0.0090077 ]),
 'mean_score_time': array([0.04601987, 0.07810728, 0.05841215, 0.06710513, 0.07253893,
        0.06024845, 0.06239263, 0.07082645, 0.07137823, 0.03139591,
        0.06834642, 0.05967283]),
 'std_score_time': array([0.00789648, 0.00209781, 0.00179199, 0.03224923, 0.0025285 ,
        0.00058856, 0.04011324, 0.00094527, 0.02853212, 0.00098106,
        0.00254595, 0.00142764]),
 'param_class_weight': masked_array(data=['balanced', 'balanced', 'balanced', 'balanced',
                    'balanced', 'balanced', None, None, None, None, None,
                    None],
              mask=[False, False, False, False,

In [75]:
results_svm = pd.DataFrame(grid.cv_results_)

In [76]:
results_svm = results_svm.sort_values(by='rank_test_score')

In [77]:
results_svm

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
6,19.523406,3.771570,0.062393,0.040113,None,scale,linear,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.648889,0.670379,0.659243,0.659503,0.008775,1
9,15.673601,4.002509,0.031396,0.000981,None,auto,linear,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.648889,0.670379,0.659243,0.659503,0.008775,1
0,18.378194,1.545768,0.046020,0.007896,balanced,scale,linear,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.640000,0.625835,0.608018,0.624618,0.013085,3
3,18.280367,2.457030,0.067105,0.032249,balanced,auto,linear,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.640000,0.625835,0.608018,0.624618,0.013085,3
10,0.743109,0.004772,0.068346,0.002546,None,auto,rbf,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.533333,0.552339,0.478842,0.521505,0.031149,5
4,0.868346,0.015562,0.072539,0.002528,balanced,auto,rbf,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.491111,0.525612,0.481069,0.499264,0.019077,6
7,0.685309,0.004096,0.070826,0.000945,None,scale,rbf,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.326667,0.340757,0.318486,0.328636,0.009198,7
1,0.877339,0.013361,0.078107,0.002098,balanced,scale,rbf,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.264444,0.305122,0.280624,0.283397,0.016722,8
8,0.695761,0.013145,0.071378,0.028532,None,scale,sigmoid,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.237778,0.222717,0.171492,0.210662,0.028372,9
2,0.895574,0.034130,0.058412,0.001792,balanced,scale,sigmoid,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.228889,0.224944,0.162584,0.205472,0.030370,10


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [78]:
dt = DecisionTreeClassifier(random_state=21)
params = {
    'max_depth': np.arange(1, 52, 10),
    'class_weight': ('balanced', None),
    'criterion': ('entropy', 'gini')
}
grid = GridSearchCV(estimator=dt, 
                    param_grid=params, 
                    cv=3,
                    scoring='accuracy',
                    verbose=2)

grid.fit(X_train, y_train)
print(f'Best score: {grid.best_score_}')
print(f'Best parameters: {grid.best_params_}')

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END class_weight=balanced, criterion=entropy, max_depth=1; total time=   0.1s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=11; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=11; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=11; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=21; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=21; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=21; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=31; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=31; total time=   0.0s
[CV] END class_wei

Best score: 0.8590579889466303

Best parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': 31}

In [79]:
results_dt = pd.DataFrame(grid.cv_results_)
results_dt = results_svm.sort_values(by='rank_test_score')
results_dt

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
6,19.523406,3.771570,0.062393,0.040113,None,scale,linear,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.648889,0.670379,0.659243,0.659503,0.008775,1
9,15.673601,4.002509,0.031396,0.000981,None,auto,linear,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.648889,0.670379,0.659243,0.659503,0.008775,1
0,18.378194,1.545768,0.046020,0.007896,balanced,scale,linear,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.640000,0.625835,0.608018,0.624618,0.013085,3
3,18.280367,2.457030,0.067105,0.032249,balanced,auto,linear,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.640000,0.625835,0.608018,0.624618,0.013085,3
10,0.743109,0.004772,0.068346,0.002546,None,auto,rbf,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.533333,0.552339,0.478842,0.521505,0.031149,5
4,0.868346,0.015562,0.072539,0.002528,balanced,auto,rbf,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.491111,0.525612,0.481069,0.499264,0.019077,6
7,0.685309,0.004096,0.070826,0.000945,None,scale,rbf,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.326667,0.340757,0.318486,0.328636,0.009198,7
1,0.877339,0.013361,0.078107,0.002098,balanced,scale,rbf,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.264444,0.305122,0.280624,0.283397,0.016722,8
8,0.695761,0.013145,0.071378,0.028532,None,scale,sigmoid,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.237778,0.222717,0.171492,0.210662,0.028372,9
2,0.895574,0.034130,0.058412,0.001792,balanced,scale,sigmoid,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.228889,0.224944,0.162584,0.205472,0.030370,10


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [80]:
rf = RandomForestClassifier(random_state=21)
params = {
    'n_estimators': (5, 10, 50, 100),
    'max_depth': np.arange(1, 52, 10),
    'class_weight': ('balanced', None),
    'criterion': ('entropy', 'gini')
}
grid = GridSearchCV(estimator=rf, 
                    param_grid=params, 
                    cv=3,
                    scoring='accuracy',
                    verbose=2)

grid.fit(X_train, y_train)
print(f'Best score: {grid.best_score_}')
print(f'Best parameters: {grid.best_params_}')

Fitting 3 folds for each of 96 candidates, totalling 288 fits
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=5; total time=   0.1s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=5; total time=   0.1s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=5; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=10; total time=   0.1s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=10; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=10; total time=   0.0s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=50; total time=   0.2s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=50; total time=   0.2s
[CV] END class_weight=balanced, criterion=entropy, max_depth=1, n_estimators=50; total time=   0.2s
[CV] END class_weight=balanced, criterion

[CV] END class_weight=balanced, criterion=gini, max_depth=1, n_estimators=100; total time=   0.7s
[CV] END class_weight=balanced, criterion=gini, max_depth=1, n_estimators=100; total time=   0.4s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=5; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=5; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=5; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=10; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=10; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=10; total time=   0.1s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=50; total time=   0.5s
[CV] END class_weight=balanced, criterion=gini, max_depth=11, n_estimators=50; total time=   0.3s
[CV] END class_weight=b

[CV] END class_weight=None, criterion=entropy, max_depth=11, n_estimators=100; total time=   0.5s
[CV] END class_weight=None, criterion=entropy, max_depth=11, n_estimators=100; total time=   0.4s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=10; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=10; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=10; total time=   0.0s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=50; total time=   0.2s
[CV] END class_weight=None, criterion=entropy, max_depth=21, n_estimators=50; total time=   0.2s
[CV] END class_weight=None, cri

[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=10; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=50; total time=   0.2s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=50; total time=   0.2s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=50; total time=   0.2s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=100; total time=   0.3s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=100; total time=   0.3s
[CV] END class_weight=None, criterion=gini, max_depth=31, n_estimators=100; total time=   0.3s
[CV] END class_weight=None, criterion=gini, max_depth=41, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=41, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=41, n_estimators=5; total time=   0.0s
[CV] END class_weight=None, criterion=gini, max_depth=41, n_

Best score: 0.8590579889466303

Best parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': 31}

In [81]:
results_rf = pd.DataFrame(grid.cv_results_)
results_rf = results_svm.sort_values(by='rank_test_score')
results_rf

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
6,19.523406,3.771570,0.062393,0.040113,None,scale,linear,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.648889,0.670379,0.659243,0.659503,0.008775,1
9,15.673601,4.002509,0.031396,0.000981,None,auto,linear,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.648889,0.670379,0.659243,0.659503,0.008775,1
0,18.378194,1.545768,0.046020,0.007896,balanced,scale,linear,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.640000,0.625835,0.608018,0.624618,0.013085,3
3,18.280367,2.457030,0.067105,0.032249,balanced,auto,linear,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.640000,0.625835,0.608018,0.624618,0.013085,3
10,0.743109,0.004772,0.068346,0.002546,None,auto,rbf,"{'class_weight': None, 'gamma': 'auto', 'kerne...",0.533333,0.552339,0.478842,0.521505,0.031149,5
4,0.868346,0.015562,0.072539,0.002528,balanced,auto,rbf,"{'class_weight': 'balanced', 'gamma': 'auto', ...",0.491111,0.525612,0.481069,0.499264,0.019077,6
7,0.685309,0.004096,0.070826,0.000945,None,scale,rbf,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.326667,0.340757,0.318486,0.328636,0.009198,7
1,0.877339,0.013361,0.078107,0.002098,balanced,scale,rbf,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.264444,0.305122,0.280624,0.283397,0.016722,8
8,0.695761,0.013145,0.071378,0.028532,None,scale,sigmoid,"{'class_weight': None, 'gamma': 'scale', 'kern...",0.237778,0.222717,0.171492,0.210662,0.028372,9
2,0.895574,0.034130,0.058412,0.001792,balanced,scale,sigmoid,"{'class_weight': 'balanced', 'gamma': 'scale',...",0.228889,0.224944,0.162584,0.205472,0.030370,10


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [82]:
results = []

for n_estimators in tqdm([5, 10, 50, 100], desc='n_estimators'):
    for max_depth in np.arange(1, 52, 10):
        for class_weight in ('balanced', None):
            for criterion in ('entropy', 'gini'):

                rf = RandomForestClassifier(random_state=21,
                                            n_estimators=n_estimators,
                                            max_depth=max_depth, 
                                            class_weight=class_weight,
                                            criterion=criterion)
                scores = cross_val_score(rf, X_train, y_train, cv=5, n_jobs=-1)
                
                results.append({
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'class_weight': class_weight,
                    'criterion': criterion,
                    'mean_accuracy': scores.mean(),
                    'std_accuracy': scores.std()
                })

n_estimators: 100%|██████████| 4/4 [01:19<00:00, 19.81s/it]


In [83]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='mean_accuracy', ascending=False)
results_df

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
87,100,31,None,gini,0.903547,0.014380
95,100,51,None,gini,0.902806,0.010460
91,100,41,None,gini,0.902806,0.010460
65,50,41,balanced,gini,0.902065,0.014082
69,50,51,balanced,gini,0.902065,0.014082
...,...,...,...,...,...,...
26,10,1,None,entropy,0.369404,0.019380
3,5,1,None,gini,0.364219,0.021651
2,5,1,None,entropy,0.353832,0.016467
1,5,1,balanced,gini,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [84]:
rf = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=31, 
                            class_weight=None,
                            criterion='gini')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

In [85]:
accuracy_score(y_test, pred)    

0.9378698224852071